In [1]:
import subprocess, sys
import torch

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

device = torch.device('cuda:0')
print(f'\nUsing device: {device}')

! nvidia-smi

Python: 3.10.20
PyTorch: 2.6.0+cu124
CUDA: <module 'torch.cuda' from '/home/samofforjindu_lfis/anaconda3/envs/brats-env/lib/python3.10/site-packages/torch/cuda/__init__.py'>
GPU: NVIDIA L4

Using device: cuda:0
Tue May 19 07:21:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8    

In [4]:
import sys
import logging
from pathlib import Path

# Project root
PROJECT_ROOT = Path('/home/samofforjindu_lfis/brats-research')
# PROJECT_ROOT = Path('C:/Users/sammi/Desktop/projects/brats-2023')

assert PROJECT_ROOT.exists(), f'PROJECT_ROOT not found: {PROJECT_ROOT}'

# Add project root to path so `from datasets import ...` works
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Each challenge lives in its own extracted subfolder under DATASET_DIR
DATASET_DIR = PROJECT_ROOT / 'dataset'
assert DATASET_DIR.exists(), f'DATASET_DIR not found: {DATASET_DIR}'

In [ ]:
# Keys = short challenge names used throughout the codebase
# Values = actual extracted directory names (from the zip archives)
CHALLENGE_DIRS = {
    'GLI': DATASET_DIR / 'ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData',
    'MEN': DATASET_DIR / 'ASNR-MICCAI-BraTS2023-MEN-Challenge-TrainingData',
    'PED': DATASET_DIR / 'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData',
}

CHALLENGE_DIRS.items()

In [ ]:
# Output directories
SPLITS_DIR = PROJECT_ROOT / 'splits'
CACHE_DIR = PROJECT_ROOT / 'cache'

SPLITS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)

# Module imports
from loader import (
    BraTSDataset,
    SplitManager,
    DatasetVerifier,
    MetadataExtractor,
)

print('All imports OK.')
print(f'Splits → {SPLITS_DIR}')
print(f'Cache → {CACHE_DIR}')

In [ ]:
# Expected subject name prefix per challenge
PREFIXES = {
    'GLI': 'BraTS-GLI',
    'MEN': 'BraTS-MEN',
    'PED': 'BraTS-PED'
}

MODALITIES = ('t1c', 't1n', 't2f', 't2w')

scan_results = {}

for challenge, data_dir in CHALLENGE_DIRS.items():
    prefix = PREFIXES[challenge]

    if not data_dir.exists():
        print(f'[{challenge}]  ✗  Directory not found: {data_dir}')
        scan_results[challenge] = {'found': False}

        continue

    subject_dirs = sorted(
        d for d in data_dir.iterdir()
        if d.is_dir() and d.name.startswith(prefix)
    )

    complete, incomplete = [], []

    for sd in subject_dirs:
        has_all_mods = all((sd / f'{sd.name}-{m}.nii.gz').exists() for m in MODALITIES)
        has_seg = (sd / f'{sd.name}-seg.nii.gz').exists()

        if has_all_mods and has_seg:
            complete.append(sd.name)
        else:
            missing = [m for m in MODALITIES if not (sd / f'{sd.name}-{m}.nii.gz').exists()]
            if not has_seg: missing.append('seg')
            incomplete.append((sd.name, missing))

    scan_results[challenge] = {
        'found': True,
        'total': len(subject_dirs),
        'complete': len(complete),
        'incomplete': incomplete,
        'subjects': complete,
    }

    status = '✓' if not incomplete else f'⚠  {len(incomplete)} incomplete'

    print(f'[{challenge}]  {status}')
    print(f' Total subject dirs : {len(subject_dirs)}')
    print(f' Complete (4 mods + seg) : {len(complete)}')

    if incomplete:
        for name, missing in incomplete[:5]:
            print(f'✗ {name} missing: {missing}')

        if len(incomplete) > 5:
            print(f'... and {len(incomplete) - 5} more')

    print()

## 3. MEN Patch Verification

In [ ]:
# The BraTS-MEN-TRAIN-FIX-V4.zip must be extracted DIRECTLY into the MEN
# training directory (allowing overwrites) before generating splits.
# This cell confirms whether that has been done.

men_dir   = CHALLENGE_DIRS['MEN']
patch_zip = DATASET_DIR / 'BraTS-MEN-TRAIN-FIX-V4.zip'

print('MEN Patch Status')
print('─' * 40)

if not men_dir.exists():
    print('✗  MEN training directory not found — extract the zip first.')
else:
    if patch_zip.exists():
        print('⚠  BraTS-MEN-TRAIN-FIX-V4.zip is still present as a zip.')
        print('   Extract it into:', men_dir)
        print('   Command (PowerShell):')
        print(f'   Expand-Archive -Path "{patch_zip}" -DestinationPath "{men_dir}" -Force')
    else:
        print('✓  Patch zip not sitting alongside data (expected after extraction).')

    # Check for a V4 subdirectory (zip extracted into subfolder instead of inline)
    v4_subdir = men_dir / 'BraTS-MEN-TRAIN-FIX-V4'
    if v4_subdir.exists():
        print('✗  Patch was extracted into a subdirectory instead of merged inline.')
        print(f'   Move contents of {v4_subdir.name}/ up one level.')
    else:
        print('✓  No patch subdirectory found — patch appears merged correctly.')

print()
print('NOTE: After applying the patch, re-run cells 2 and onwards.')

## 4. Dataset Integrity Verification

In [ ]:
# Runs shape, spacing, and label checks on each challenge.
# Label checks are sampled (50 subjects per challenge) to avoid loading
# every seg file — the shape/spacing checks cover all subjects.
#
# Expected output: zero errors, possibly a few warnings for non-standard spacing.
# Fix any errors before proceeding to split generation.

for challenge, data_dir in CHALLENGE_DIRS.items():
    if not data_dir.exists():
        print(f'[{challenge}] Skipping — directory not found.')
        continue

    print(f'\nVerifying {challenge} ...')
    report = DatasetVerifier.run(
        data_dir=data_dir,
        challenge=challenge,
        check_shapes=True,
        check_spacing=True,
        check_labels=True,
        check_men_patch=(challenge == 'MEN'),
        max_label_check_subjects=50,
    )
    print(report)

## 5. Per-Subject Metadata Extraction

In [ ]:
# Computes per-subject tumour volumes (NCR, ED, ET, total), ET fraction,
# brain voxel counts, and per-modality intensity statistics.
#
# This takes several minutes per challenge on first run (reads every seg file).
# Results are cached as JSON — subsequent runs load instantly.
#
# The metadata is used for:
#   - Stratified split generation (volume quartile + ET presence)
#   - PED adaptive ET loss threshold computation
#   - Class imbalance analysis and loss weight derivation

metadata = {}

for challenge, data_dir in CHALLENGE_DIRS.items():
    if not data_dir.exists():
        print(f'[{challenge}] Skipping — directory not found.')
        continue

    meta_path = SPLITS_DIR / f'{challenge}_metadata.json'

    if meta_path.exists():
        print(f'[{challenge}] Loading cached metadata from {meta_path.name} ...')
        metadata[challenge] = MetadataExtractor.load(meta_path)
    else:
        print(f'[{challenge}] Extracting metadata (first run — reads all seg files) ...')
        metadata[challenge] = MetadataExtractor.build_cache(
            data_dir=data_dir,
            challenge=challenge,
            output_path=meta_path,
        )
        print(f'[{challenge}] Metadata saved → {meta_path}')

print('\nMetadata loaded for:', list(metadata.keys()))

## 6. Dataset Statistics — Class Imbalance & Volume Distributions

In [ ]:
import numpy as np

for challenge, meta in metadata.items():
    stats   = meta['dataset_stats']
    n_subjs = stats['n_subjects_with_seg']

    print(f'\n{"═"*55}')
    print(f'  {challenge}  ({n_subjs} labelled subjects)')
    print(f'{"─"*55}')

    # ── Volume distribution
    vs = stats['tumour_volume_stats']

    if vs:
        print(f'  Tumour volume (voxels):')
        print(f'    Min / Q1 / Median / Q3 / Max')
        print(f'    {vs["min"]:,} / {vs["q1"]:,} / {vs["median"]:,} / {vs["q3"]:,} / {vs["max"]:,}')
        print(f'    Mean ± Std : {vs["mean"]:,} ± {vs["std"]:,}')

    # ── Class imbalance
    total_brain = stats['total_brain_voxels']
    print(f'\n  Class imbalance (all-subject voxel totals):')

    rows = [
        ('Background', total_brain - stats['total_tumour_voxels']),
        ('NCR (label 1)', stats['total_ncr_voxels']),
        ('ED  (label 2)', stats['total_ed_voxels']),
        ('ET  (label 3)', stats['total_et_voxels']),
    ]

    for label, count in rows:
        pct = 100 * count / total_brain if total_brain > 0 else 0
        bar = '█' * int(pct * 1.5)
        print(f'    {label:<18} {count:>14,} vox  ({pct:5.2f}%)  {bar}')

    # ── ET-absent subjects (critical for PED)
    et_absent = stats['et_absent_count']
    print(f'\n  ET-absent subjects: {et_absent} / {n_subjs} ({stats["et_absent_fraction"]*100:.1f}%)')

    if challenge == 'PED' and et_absent > 0:
        thresh = MetadataExtractor.compute_et_threshold(meta, percentile=10)
        print(f'  PED adaptive ET threshold (10th percentile of ET+ volumes): {thresh:,} voxels')

print(f'\n{"═"*55}')

## 7. Stratified 5-Fold Split Generation

In [ ]:
# Generates one JSON split file per challenge.
# These files are the single source of truth for all train/val splits.
# Commit them to git — never regenerate during experiments.
#
# Stratification is on:
#   - Tumour volume quartile  (ensures small/large tumours are balanced per fold)
#   - ET presence (binary)    (critical for PED where ET can be absent)
#
# Set REGENERATE = True only if you deliberately want to rebuild the splits
# (e.g. after applying the MEN patch or adding new subjects).

REGENERATE = False

splits = {}

for challenge, data_dir in CHALLENGE_DIRS.items():
    if not data_dir.exists():
        print(f'[{challenge}] Skipping — directory not found.')
        continue

    split_path = SPLITS_DIR / f'{challenge}_5fold_split.json'

    if split_path.exists() and not REGENERATE:
        print(f'[{challenge}] Split already exists — loading {split_path.name}')
        splits[challenge] = SplitManager.load(split_path)
    else:
        print(f'[{challenge}] Generating stratified 5-fold split ...')
        splits[challenge] = SplitManager.generate(
            data_dir=data_dir,
            challenge=challenge,
            output_path=split_path,
            n_folds=5,
            seed=42,
            overwrite=REGENERATE,
        )
        print(f'[{challenge}] Split saved → {split_path}')

print('\nDone. Split files:')
for challenge in splits:
    p = SPLITS_DIR / f'{challenge}_5fold_split.json'
    print(f'  {p}')

## 8. Validate Splits — Fold Balance & Stratification Quality

In [ ]:
# Verifies that:
#   1. Train+val counts are consistent across folds
#   2. Tumour volume distribution is similar in each fold
#   3. ET-absent subjects are spread evenly (critical for PED)

for challenge, split_data in splits.items():
    print(SplitManager.summary(split_data))

    # Per-fold volume quartile distribution
    subject_meta = split_data['subjects']
    print(f'\n  Per-fold quartile distribution (Q1=smallest tumours):')
    print(f'  {"Fold":<6} {"Q1":>5} {"Q2":>5} {"Q3":>5} {"Q4":>5} {"ET-absent":>10}')
    print(f'  {"-"*42}')

    for fold_key, fold_data in split_data['folds'].items():
        val_subjects = fold_data['val']
        quartiles    = [0, 0, 0, 0]
        et_absent    = 0

        for s in val_subjects:
            if s not in subject_meta:
                continue
            q = subject_meta[s].get('volume_quartile', 0)
            if 1 <= q <= 4:
                quartiles[q-1] += 1
            if subject_meta[s].get('et_volume_voxels', 1) == 0:
                et_absent += 1
        print(f'  {fold_key:<6} {quartiles[0]:>5} {quartiles[1]:>5} {quartiles[2]:>5} {quartiles[3]:>5} {et_absent:>10}')
    print()

## 9. BraTSDataset Smoke Test — Load One Subject Per Challenge

In [ ]:
import time

# Instantiate a val dataset for fold 0 of each challenge and load one subject.
# Confirms the end-to-end pipeline (scan → preprocess → crop → tensor) works.

for challenge, data_dir in CHALLENGE_DIRS.items():
    if not data_dir.exists() or challenge not in splits:
        print(f'[{challenge}] Skipping.')
        continue

    split_path = SPLITS_DIR / f'{challenge}_5fold_split.json'

    ds = BraTSDataset(
        data_dir   = data_dir,
        challenge  = challenge,
        mode       = 'val',
        fold       = 0,
        split_path = split_path,
        # No transform for smoke test — raw preprocessed tensors only
    )
    print(ds)

    t0 = time.perf_counter()
    name, imgs, seg = ds[0]
    elapsed = time.perf_counter() - t0

    print(f'  Subject    : {name}')
    print(f'  Load time  : {elapsed:.2f}s')
    print(f'  Modalities : {len(imgs)} tensors, each shape {tuple(imgs[0].shape)}')
    print(f'  Seg shape  : {tuple(seg.shape)}')
    print(f'  Seg labels : {seg.unique().tolist()}')

    # Check intensity range of first modality
    t1c = imgs[0]
    print(f'  T1c range  : [{t1c.min():.1f}, {t1c.max():.1f}]  '
          f'mean={t1c[t1c>0].mean():.1f}  std={t1c[t1c>0].std():.1f}')
    print()

## 10. DataLoader Smoke Test

In [ ]:
from torch.utils.data import DataLoader

# Verify the DataLoader collation works correctly for the list-of-tensors
# return format expected by the existing training loop.

challenge  = 'GLI'   # Use GLI for the DataLoader test (largest dataset)
data_dir   = CHALLENGE_DIRS[challenge]
split_path = SPLITS_DIR / f'{challenge}_5fold_split.json'

ds_train = BraTSDataset(
    data_dir   = data_dir,
    challenge  = challenge,
    mode       = 'train',
    fold       = 0,
    split_path = split_path,
    cache_dir  = CACHE_DIR / challenge,
)

loader = DataLoader(
    ds_train,
    batch_size   = 2,
    shuffle      = True,
    num_workers  = 4,
    pin_memory   = True,
    persistent_workers = True,
)

print(f'Dataset : {ds_train}')
print(f'Batches : {len(loader)}')
print()

# Pull one batch and verify shapes
t0 = time.perf_counter()
names, imgs, seg = next(iter(loader))
elapsed = time.perf_counter() - t0

import torch
x_in = torch.cat(imgs, dim=1)   # Simulate the training loop cat

print(f'First batch loaded in {elapsed:.2f}s')
print(f'  names    : {names}')
print(f'  imgs[0]  : {tuple(imgs[0].shape)}   (B, 1, H, W, D)')
print(f'  x_in     : {tuple(x_in.shape)}   (B, 4, H, W, D)  ← model input')
print(f'  seg      : {tuple(seg.shape)}   (B, 1, H, W, D)')
print(f'  seg vals : {seg.unique().tolist()}')

## 11. EDA — Volume Distribution Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

fig = plt.figure(figsize=(18, 5 * len(metadata)))
gs  = gridspec.GridSpec(len(metadata), 3, figure=fig, hspace=0.45, wspace=0.35)

for row_i, (challenge, meta) in enumerate(metadata.items()):
    subjects = meta['subjects']

    total_vols = np.array([v['tumour_volume_voxels'] for v in subjects.values()])
    et_vols    = np.array([v['et_volume_voxels']     for v in subjects.values()])
    et_frac    = np.array([v['et_fraction']          for v in subjects.values()])

    # ── Plot 1: Total tumour volume histogram ─────────────────────────────
    ax1 = fig.add_subplot(gs[row_i, 0])
    ax1.hist(total_vols, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    ax1.axvline(np.median(total_vols), color='tomato', linestyle='--', label=f'Median: {int(np.median(total_vols)):,}')
    ax1.set_title(f'{challenge} — Total Tumour Volume')
    ax1.set_xlabel('Voxels')
    ax1.set_ylabel('Subjects')
    ax1.legend(fontsize=8)

    # ── Plot 2: ET volume histogram ───────────────────────────────────────
    ax2 = fig.add_subplot(gs[row_i, 1])
    et_pos = et_vols[et_vols > 0]
    et_absent_n = int((et_vols == 0).sum())
    ax2.hist(et_pos, bins=40, color='darkorange', edgecolor='white', linewidth=0.4)
    ax2.axvline(np.median(et_pos) if len(et_pos) else 0, color='tomato',
                linestyle='--', label=f'Median: {int(np.median(et_pos)) if len(et_pos) else 0:,}')
    ax2.set_title(f'{challenge} — ET Volume  ({et_absent_n} ET-absent)')
    ax2.set_xlabel('Voxels (ET > 0 subjects only)')
    ax2.set_ylabel('Subjects')
    ax2.legend(fontsize=8)

    # ── Plot 3: Region composition stacked bar (per-subject sample) ───────
    ax3 = fig.add_subplot(gs[row_i, 2])
    ncr_v = np.array([v['ncr_volume_voxels'] for v in subjects.values()], dtype=float)
    ed_v  = np.array([v['ed_volume_voxels']  for v in subjects.values()], dtype=float)
    et_v  = np.array([v['et_volume_voxels']  for v in subjects.values()], dtype=float)

    # Normalise to fractions
    total = ncr_v + ed_v + et_v
    total = np.where(total == 0, 1, total)   # avoid div-by-zero
    order = np.argsort(total)[::-1][:100]    # top-100 by volume for readability
    xs    = np.arange(len(order))

    ax3.bar(xs, ncr_v[order] / total[order], label='NCR', color='#4c72b0')
    ax3.bar(xs, ed_v[order]  / total[order], bottom=ncr_v[order]/total[order],
            label='ED',  color='#dd8452')
    ax3.bar(xs, et_v[order]  / total[order],
            bottom=(ncr_v[order]+ed_v[order])/total[order],
            label='ET',  color='#55a868')
    ax3.set_title(f'{challenge} — Region Composition (top 100 by volume)')
    ax3.set_xlabel('Subject rank (largest tumour first)')
    ax3.set_ylabel('Fraction of tumour volume')
    ax3.legend(fontsize=8, loc='lower right')

plt.suptitle('BraTS 2023 — Dataset EDA', fontsize=14, y=1.01)
plt.savefig(SPLITS_DIR / 'dataset_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Figure saved → {SPLITS_DIR / "dataset_eda.png"}')

## 12. EDA — Visualise One Subject (Images + Segmentation)

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def visualise_subject(
    data_dir: Path,
    subject_name: str,
    slice_axis: int = 2,
    n_slices: int = 5,
):
    """
    Visualise n_slices evenly spaced slices for one subject.
    Rows: T1c, T1n, T2f, T2w, Segmentation.

    Parameters
    ----------
    slice_axis : int
        0=sagittal, 1=coronal, 2=axial (default)
    n_slices : int
        Number of slices to display side-by-side.
    """
    subject_dir = data_dir / subject_name
    MODALITIES  = ['t1c', 't1n', 't2f', 't2w']

    # Load raw volumes (no preprocessing — shows actual scanner data)
    vols = {}
    for mod in MODALITIES:
        path = subject_dir / f'{subject_name}-{mod}.nii.gz'
        vols[mod] = nib.load(str(path)).get_fdata(dtype=np.float32)

    seg_path = subject_dir / f'{subject_name}-seg.nii.gz'
    seg = nib.load(str(seg_path)).get_fdata(dtype=np.float32) if seg_path.exists() else None

    # Find slices with most tumour content
    vol_shape = vols['t1c'].shape
    if seg is not None:
        tumour_counts = np.sum(seg > 0, axis=tuple(i for i in range(3) if i != slice_axis))
        top_indices   = np.argsort(tumour_counts)[::-1]
        # Pick n_slices evenly spaced from the top 80%
        top_n  = max(int(0.8 * np.sum(tumour_counts > 0)), n_slices)
        chosen = np.sort(top_indices[:top_n][
            np.linspace(0, top_n - 1, n_slices, dtype=int)
        ])
    else:
        chosen = np.linspace(vol_shape[slice_axis] // 4,
                             3 * vol_shape[slice_axis] // 4,
                             n_slices, dtype=int)

    n_rows = len(MODALITIES) + (1 if seg is not None else 0)
    fig, axes = plt.subplots(n_rows, n_slices, figsize=(3 * n_slices, 3 * n_rows))
    if n_slices == 1:
        axes = axes[:, np.newaxis]

    # Segmentation colour map: 0=bg, 1=NCR(blue), 2=ED(yellow), 3=ET(red)
    seg_cmap   = mcolors.ListedColormap(['black', '#2196F3', '#FFC107', '#F44336'])
    seg_norm   = mcolors.BoundaryNorm([0, 0.5, 1.5, 2.5, 3.5], seg_cmap.N)

    axis_label = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}[slice_axis]

    for col, sl_idx in enumerate(chosen):
        for row, mod in enumerate(MODALITIES):
            v = vols[mod]
            slc = np.take(v, sl_idx, axis=slice_axis)
            axes[row, col].imshow(slc.T, cmap='gray', origin='lower',
                                  vmin=np.percentile(v[v>0], 1) if v.any() else 0,
                                  vmax=np.percentile(v[v>0], 99) if v.any() else 1)
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(mod.upper(), fontsize=10)
            if row == 0:
                axes[row, col].set_title(f'{axis_label} {sl_idx}', fontsize=9)

        if seg is not None:
            seg_slc = np.take(seg, sl_idx, axis=slice_axis)
            axes[n_rows-1, col].imshow(seg_slc.T, cmap=seg_cmap, norm=seg_norm, origin='lower')
            axes[n_rows-1, col].axis('off')
            if col == 0:
                axes[n_rows-1, col].set_ylabel('SEG', fontsize=10)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2196F3', label='NCR (label 1)'),
        Patch(facecolor='#FFC107', label='ED  (label 2)'),
        Patch(facecolor='#F44336', label='ET  (label 3)'),
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.01))

    fig.suptitle(f'{subject_name}', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


# ── Visualise one subject from each challenge ─────────────────────────────────
for challenge, split_data in splits.items():
    data_dir = CHALLENGE_DIRS[challenge]
    if not data_dir.exists():
        continue

    # Pick the subject with the largest tumour for a clear visualisation
    subject_meta = split_data['subjects']
    largest = max(
        subject_meta.items(),
        key=lambda kv: kv[1].get('tumour_volume_voxels', 0)
    )[0]

    print(f'[{challenge}] Visualising: {largest}')
    visualise_subject(data_dir, largest, slice_axis=2, n_slices=5)

## 13. Summary & Next Steps

After this notebook runs cleanly end-to-end:

| File | Purpose |
|---|---|
| `splits/{CHALLENGE}_metadata.json` | Per-subject volumes, ET fractions, intensity stats |
| `splits/{CHALLENGE}_5fold_split.json` | 5-fold stratified CV split (commit to git) |
| `splits/dataset_eda.png` | Volume + class imbalance overview figure |

**Commit `splits/*.json` to git now.** These files are the single source of truth for all train/val splits across every experiment.

**Next notebook:** `02_preprocessing.ipynb` — augmentation pipeline, MONAI transforms, and patch-based DataLoader configuration.